# 3. Use a dataset with a PyTorch DataLoader

ESP datasets are map-style — they implement `__len__` and `__getitem__`, so they plug directly into `torch.utils.data.DataLoader`.

In [2]:
# Add torch dependency
!uv pip install torch

Resolved 10 packages in 2.24s                                        
Prepared 3 packages in 13.60s                                            
Installed 10 packages in 240ms                              
 + filelock==3.29.0
 + fsspec==2026.3.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + setuptools==81.0.0
 + sympy==1.14.0
 + torch==2.11.0
 + typing-extensions==4.15.0


In [3]:
import torch
from torch.utils.data import DataLoader
from alp_data import Beans

dataset = Beans(split="dogs_test", sample_rate=16000)
print(f"Dataset length: {len(dataset)}")

Dataset length: 139


Audio clips have variable length. We'll write a small `collate_fn` that pads to the longest in the batch and keeps just the fields we care about.

In [4]:
def collate_pad(batch):
    waves = [torch.as_tensor(s["audio"], dtype=torch.float32) for s in batch]
    lengths = torch.tensor([w.shape[-1] for w in waves])
    max_len = int(lengths.max())
    padded = torch.zeros(len(waves), max_len, dtype=torch.float32)
    for i, w in enumerate(waves):
        padded[i, : w.shape[-1]] = w
    labels = [s.get("label") for s in batch]
    return {"audio": padded, "lengths": lengths, "label": labels}

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_pad,
)

In [5]:
for i, batch in enumerate(loader):
    print(f"batch {i}: audio={tuple(batch['audio'].shape)}, lengths={batch['lengths'].tolist()}, labels={batch['label']}")
    if i >= 2:
        break

batch 0: audio=(4, 191074), lengths=[121253, 191074, 181463, 129261], labels=['Mac', 'Zoe', 'Farley', 'Siggy']
batch 1: audio=(4, 196116), lengths=[133235, 29068, 196116, 91414], labels=['Mac', 'Luke', 'Farley', 'Roodie']
batch 2: audio=(4, 302715), lengths=[252885, 302715, 144446, 62643], labels=['Farley', 'Louie', 'Keri', 'Roodie']


⚠️ Dataset class does not currently work with torch IterableDataset type